# A ConvNet on MNIST, and why it beats the dense model

The same problem as chapter 2, with a fraction of the parameters and a better score. The reason is two properties of the convolution operation, and both are visible in the numbers.

**Runs on:** CPU — about 3 minutes (GPU: 30 seconds) &nbsp;·&nbsp; **Slides:** [Chapter 8 — Image Classification](../../../course-web-slides/ch08/index.html) &nbsp;·&nbsp; **Section:** 01 — Introduction to convnets

---

## The model

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Expected output:

```
Total params: 104,202
```

## Training it

In [ ]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28, 28, 1)).astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype("float32") / 255

model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(train_images, train_labels, epochs=5, batch_size=64, verbose=2)
_, test_acc = model.evaluate(test_images, test_labels, verbose=0)
print(f"test accuracy: {test_acc:.4f}")

Expected output:

```
test accuracy: 0.99xx
```

## Against the dense model from chapter 2

In [ ]:
dense = keras.Sequential([layers.Flatten(),
                          layers.Dense(512, activation="relu"),
                          layers.Dense(10, activation="softmax")])
dense.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
dense.fit(train_images, train_labels, epochs=5, batch_size=64, verbose=0)
_, dense_acc = dense.evaluate(test_images, test_labels, verbose=0)

print(f"{'model':10s} {'params':>10s} {'test acc':>10s}   error rate")
print(f"{'dense':10s} {dense.count_params():>10,} {dense_acc:>10.4f}   "
      f"{1-dense_acc:.4f}")
print(f"{'convnet':10s} {model.count_params():>10,} {test_acc:>10.4f}   "
      f"{1-test_acc:.4f}")
print(f"\nerror reduced by {(1-dense_acc)/(1-test_acc):.1f}x, "
      f"with {dense.count_params()/model.count_params():.1f}x fewer parameters")

**Fewer parameters and a lower error.** Two properties of convolution do this:

- **Translation invariance** — a pattern learned in one corner is recognised everywhere, so the model does not relearn it 784 times.
- **Spatial hierarchies** — small local patterns compose into larger ones, layer by layer.

A Dense layer has neither, which is why it needs five times as many parameters to do worse.

## Watching the spatial dimensions shrink

In [ ]:
for layer in model.layers:
    print(f"{layer.__class__.__name__:16s} {str(layer.output.shape):24s} "
          f"{layer.count_params():>8,} params")

Height and width fall — 28, 26, 13, 11, 5, 3 — while depth rises: 1, 32, 64, 128. **Trading space for semantics**, which is the shape of every ConvNet in this course.

## Why max pooling and not something gentler

In [ ]:
def build(pooling):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, activation="relu")(i)
    if pooling == "max":
        x = layers.MaxPooling2D(2)(x)
    elif pooling == "avg":
        x = layers.AveragePooling2D(2)(x)
    elif pooling == "stride":
        x = layers.Conv2D(32, 3, strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.Flatten()(x)
    o = layers.Dense(10, activation="softmax")(x)
    m = keras.Model(i, o)
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    m.fit(train_images[:20000], train_labels[:20000], epochs=3,
          batch_size=64, verbose=0)
    return m.evaluate(test_images, test_labels, verbose=0)[1]

for p in ["max", "avg", "stride"]:
    print(f"{p:8s} pooling -> test accuracy {build(p):.4f}")

Max pooling usually wins here. The argument is that features encode the **presence** of a pattern, and averaging dilutes presence while max preserves it. Note that chapters 11 and 17 prefer strides — because segmentation and generation care *where* things are, and max pooling discards exactly that.

## What the first layer learned

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

filters = model.layers[1].get_weights()[0]      # (3, 3, 1, 32)
f = filters[:, :, 0, :]
f = (f - f.min()) / (f.max() - f.min())

fig, axes = plt.subplots(4, 8, figsize=(9, 4.6))
for ax, i in zip(axes.ravel(), range(32)):
    ax.imshow(f[:, :, i], cmap="gray"); ax.axis("off")
plt.suptitle("The 32 first-layer 3x3 filters", y=1.02)
plt.tight_layout(); plt.show()

Edge and blob detectors, learned rather than designed. Chapter 10 does this properly, at every depth, and the story it tells is the same one this glimpse suggests.

---

## What to take away

- A ConvNet beats a dense model on images with **five times fewer parameters**.
- Translation invariance and spatial hierarchies are the two reasons.
- Height and width shrink while depth grows — space traded for semantics.
- Max pooling preserves presence; use strides when position matters (chapters 11 and 17).